<img src="https://github.com/moroneyt/MXB301/raw/main/resources/qutlogo.jpg">

# MXB301 Mathematics of AI
# Lesson 11: Reverse diffusion

#### Tim Moroney, 2026

A lesson where we introduce the Ornstein-Uhlenbeck process for diffusion, and somehow manage to make it run in reverse.


# Package management

In [ ]:
import Pkg
if haskey(ENV, "COLAB_GPU") # check if we're on Colab
  if !isfile("/content/MXB301_2026_01_CPU.tgz") # check if we've already downloaded
    # download precompiled Julia environment for Colab
    run(`gdown https://drive.google.com/uc\?id=1mT9XFadzdfK8CWb5a7BYLUkd2RTi2eZc`)

    # replace Colab's Julia environment with downloaded version
    run(`rm -rf /root/.julia`)
    run(`tar -xzf MXB301_2026_01_CPU.tgz -C /root`)
  end
else
  # For any other machine we install the packages in the usual way
  Pkg.activate(".")
  Pkg.add(["CairoMakie", "CodecZlib", "ColorSchemes", "ComponentArrays", "CondaPkg",
           "DifferentiationInterface", "Distributions", "Downloads", "FiniteDiff", "ForwardDiff",
           "HTTP", "JLD2", "LaTeXStrings", "LinearAlgebra", "Lux", "MKL", "MLUtils", "NNlib",
           "NLSolversBase", "OneHotArrays", "Optim", "PythonCall", "QuadGK", "Random",
           "SpecialFunctions", "Statistics", "StatsBase", "ToeplitzMatrices", "Zygote"])
end

using CairoMakie
using DifferentiationInterface
using LaTeXStrings
using LinearAlgebra
using Lux
using Random
using SpecialFunctions
using Statistics

using ComponentArrays: ComponentVector
using Distributions: Normal, Exponential
using Downloads: download
using ForwardDiff: Dual, partials
using JLD2: jldopen
using MLUtils: DataLoader, rand_like, randn_like
using NLSolversBase: only_fg
using NNlib: softmax, sigmoid, scatter as scattergrad, conv, ∇conv_filter, ∇conv_data, DenseConvDims
using OneHotArrays: onehot, onehotbatch, onecold
using StatsBase: crossentropy, sample, Weights
using ToeplitzMatrices: Toeplitz, Hankel
using QuadGK: quadgk

import CodecZlib
import ColorSchemes
import FiniteDiff
import HTTP
import MKL
import Optim
import Zygote

# Set the random seed for reproducibility
rng = Random.seed!(0)

# SVG format scales properly in web pages and PDFs
CairoMakie.activate!(type = "svg")

# Revision of SDEs and PDEs

In the last lesson we learned about SDEs, and in particular we learned that the SDE
$$
\textrm{d}Z_{t} = v\, \textrm{d}t + \sigma\, \textrm{d}W_t
$$
models a diffusion process with drift.  The solution to this equation, subject to the initial condition $Z_0 = z_0$ is
$$
Z_t \sim \mathcal{N}(z_0 + vt, \sigma^2 t)\,.
$$
That is, $Z_t$ is a stochastic process -- a random variable indexed by time.  For any time $t$, its distribution is normal, with mean
$$
\mathbb{E}[Z_t] = z_0 + vt
$$
and variance
$$
\textrm{var}[Z_t] = \sigma^2 t\,.
$$

Like any continuous random variable, $Z_t$ has an associated probability density function $u(z,t)$.  It too, satisfies a governing equation.  For this example, the governing equation for $u$ is the [Fokker-Planck equation](https://en.wikipedia.org/wiki/Fokker_Planck_equation) with constant coefficients, a partial differential equation (PDE):
$$
\frac{\partial u}{\partial t} =  -v \frac{\partial u}{\partial z} + D \frac{\partial^2 u}{\partial z^2} \qquad (\dagger)
$$
where $D = \sigma^2 /2 $.  The solution of this PDE, subject to the initial condition $u(z,0) = \delta(z - z_0)$ is
$$
u(z|z_0,t) = \frac{1}{\sqrt{4\pi Dt}}\, \exp \left(- \frac{(z-(z_0 + vt))^2}{4Dt}  \right)\,.
$$

If instead of taking a particular initial value, $Z_0$ instead follows some probability distribution $Z_0 \sim u_0(z)$, then the best we can say about the  probability density at later times is to write it as a convolution:
$$
u(z,t) = \int_{z_0} u(z|z_0,t)\, u_0(z_0)\, \mathrm{d}z_0
$$
where $u(z|z_0,t)$ is the Gaussian we wrote above.  Perhaps in some special cases you'll be lucky and this integral can be evaluated analytically, but in general it can't be.  The solution is nonetheless a valid probability density function in all cases.


In summary, the SDE approach and the PDE approach are two complementary ways to formulate the same problem.  The object of interest in the SDE is the particle trajectory $Z_t$.  The object of interest in the PDE is the probability density function of particles, $u(z,t)$.

The SDE formulation lends itself nicely to numerical simulation, using random walks.  The PDE formulation lends itself to possibly finding closed form solutions for the probability density function, if you're lucky.



# Itô diffusion

The diffusion process above is an excellent starting point when learning about SDEs. But to reach our goal of a generative image model, we must go a little further. We require a drift term which may depend on $z$ and $t$ -- an example of so-called _Itô diffusion_:
$$
\textrm{d}Z_{t} = f(Z_t,t)\, \textrm{d}t + \sigma\, \textrm{d}W_t\qquad
$$
with initial condition
$$
Z_0 \sim u_0(z)\,.
$$
(You could also imagine $\sigma$ depending on $z$ and $t$, but we won't require that.)

The PDE satisfied by the particle's probability density function $u(z,t)$ is still called the Fokker-Planck equation
$$
\frac{\partial u}{\partial t} = -\frac{\partial}{\partial z}(f u) + D \frac{\partial^2 u} {\partial z^2}
$$
but now we must take care with the form of the drift term since $f$ is not constant. The initial condition for the PDE is
$$
u(z,0) = u_0(z)
$$
and $D = \sigma^2 /2$ as usual.

We will not prove that the Fokker-Planck equation above is correct in this more general case.  To do so requires [Itô calculus](https://en.wikipedia.org/wiki/Ito_calculus) which we don't cover in this unit.  But the general form as written here certainly generalises the result for constant coefficients in a natural way. So even without formally deriving it, we can at least be reassured that it "looks right" (which, indeed, it is).

# Forward latent "noisification"
With that essential background covered, it's time to remind ourselves of the latent image (de)noisification application that's motivating our work here.

Recall we have a variational autoencoder (VAE) that's capable of mapping an image in image space $x \in \mathcal{X}$ to a latent variable in latent space $z \in \mathcal{Z}$.  The encoder and decoder come as a pair: $q_\phi(z|x)$ and $p_\theta(x|z)$ respectively.  They are conditional probability densities, rather than deterministic mappings, to account for the inherent uncertainty in the mappings.

When training the VAE, the latent space is regularised so that the unconditional latent distribution $p(z)$ is pulled towards the standard normal distribution $\mathcal{N}(0,I)$.  That's _not_ to say that the latent distribution $p(z)$ is _indistinguishable_ from $\mathcal{N}(0,I)$.  Indeed, there is structure encoded in the way that that it differs from $\mathcal{N}(0,I)$.

For example the latent representations of real images may be more densely distributed in some regions of latent space, and more sparsely distributed in others.  There could even be whole regions of latent space where the probability density is essentially zero, even though the coordinates are perfectly consistent with an $\mathcal{N}(0,I)$ draw.

Because of this, any arbitrary latent variable $z$ drawn from $\mathcal{N}(0,I)$ may not actually correspond to a satisfactory image when decoded.  So the VAE is not, in itself, a very good _generative model_.  We can't just pick any old point from $\mathcal{N}(0,I)$, decode it, and expect to get a satisfactory image.

Ultimately our goal will be to learn a process that evolves any latent variable $z$ drawn from $\mathcal{N}(0,I)$ onto an actual latent representation of a true image.  We will find that the way to do this, is to treat the initial draw as a random variable $Z_t$ which evolves under a diffusion-type process until it is consistent with a true draw from the latent distribution $p(z)$.

In this way, we will finally be able to generate images using our latent model through the process:
1. Sample randomly from $\mathcal{N}(0,I)$
2. Apply "reverse diffusion" to evolve this sample to a true latent image
3. Apply VAE decoder to sampled latent
4. Admire generated cat picture

The process is called "reverse diffusion" because it truly does behave like diffusion in reverse.  But before we can get there, we'd better sort out the _forward_ diffusion process.  So our first goal for this lesson will be to learn how evolve a sample from the latent cat distribution into a draw from $\mathcal{N}(0,I)$ through a process of diffusion.  Afterwards we'll learn how to invert that process.

As a reminder, we built the latent cat distribution from simple formulas, in one latent dimension, specifically so that we could deal with it analytically.  In particular, we can directly sample from it.  In the bigger picture, sampling from the latent distribution is the very problem we are trying to solve.

# Ornstein-Uhlenbeck process
The particular diffusion process that we require is called the [Ornstein-Uhlenbeck process](https://en.wikipedia.org/wiki/Ornstein–Uhlenbeck_process), and in one spatial dimension its SDE takes the form

$$
\textrm{d}Z_{t} = -\theta Z_t\, \textrm{d}t + \sigma\, \textrm{d}W_t\,.\qquad
$$

You see in place of the linear drift term $v\, \textrm{d}t$ from last week, we now have the _exponential decay_ term $-\theta Z_t \textrm{d}t$.

We can understand the effect of this term if we temporarily ignore the diffusion term (set $\sigma = 0$) and examine the resulting equation
$$
\textrm{d}z = -\theta z\, \textrm{d}t
$$
which is the ordinary differential equation (ODE)
$$
\frac{\textrm{d}z}{\textrm{d}t} = -\theta z\,
$$
whose solution of course is
$$
z(t) = z_0 \textrm{e}^{-\theta t}\,.
$$



So in the Ornstein-Uhlenbeck process, the "drift" term is actually functioning as an exponential decay term.  Therefore, in the absence of a diffusion term, all trajectories would decay exponentially to zero.  But with the diffusion term included, noise is constantly being added to the process, fighting against the decay to zero.  The result of this interplay between exponential decay and diffusion is a process that can settle down to an interesting limiting distribution, as we will soon see.

The probability density function $u(z,t)$ for the Ornstein-Uhlenbeck process satisfies the Fokker-Planck equation
$$
\frac{\partial u}{\partial t} = \theta \frac{\partial}{\partial z}(z u) + D \frac{\partial^2 u} {\partial z^2}\,.\qquad (*)
$$

Reasoning statistically as we did last week, or just using good old Fourier transforms, you can solve this equation analytically (exercises!). For a delta function initial condition $u(z,0) = \delta(z - z_0)$ the solution is a Gaussian with mean
$$
\mu_t = z_0 \textrm{e}^{-\theta t}
$$
and variance
$$
s_t^2 = \frac{D}{\theta}\left(1 - \textrm{e}^{-2\theta t}\right)\,.
$$

So we do indeed see exponential decay of the mean to zero -- the same exponential decay as predicted by the analysis for the ODE above. But the variance alters the picture substantially. Here we see the interplay between $D$ (diffusivity) and $\theta$ (decay rate), with the variance ultimately limiting to
$$
s_t^2 \to \frac{D}{\theta},\qquad t \to \infty
$$

So the variance of this process remains _bounded_.

For a general initial condition $u(z,0) = u_0(z)$ we know we need to take the convolution of this Gaussian with $u_0$.  The result cannot be written in closed form in general.  But (again, using Fourier transforms), we can write down precisely what the distribution _approaches_ in the limit $t \to \infty$.  It's independent of the initial condition -- every initial condition limits to the same distribution under the Ornstein-Uhlenbeck process. And that limiting distribution is simply $\mathcal{N}\left(0, \frac{D}{\theta}\right)$.

So, for _any_ initial condition $u(z,0) = u_0(z)$, the solution to $(*)$ satisfies

$$
\lim_{t \to \infty} u(z|z_0,t) = \frac{1}{\sqrt{2\pi D/\theta}}\ \exp\left(-\frac{z^2}{2D/\theta}\right)\,.
$$


# The cat distribution

Recall we are assuming the latent cat distribution takes the form visualised below, with three peaks corresponding to sad cats, angry cats and happy cats.  We observe there is some overlap with sad cats and angry cats (sad, angry cats!) but there is no such thing as a sad, happy cat (or an angry, happy cat).  In particular the region of latent space between about $-0.5$ and $0$ has essentially zero probability.  So there is semantic structure encoded in the distribution, as we expect of our latent space.

In [ ]:
function happypdf(z, t; θ, σ, λ)
    # exponential distribution initially
    a = exp(-θ*t)
    σₜ = sqrt((σ^2/(2θ))*(1 - a^2))
    λₜ = a*λ
    ξ = z/σₜ - σₜ/λₜ
    Φ = 0.5*erfc(-ξ / sqrt(2))
    return 1/λₜ * exp(σₜ^2/(2λₜ^2) - z/λₜ) * Φ
end

function unhappypdf(z, t; θ, σ, z0, s)
    # normal distribution initially
    μₜ = z0 * exp(-θ*t)
    sₜ² = s^2 * exp(-2θ*t) + σ^2/(2θ) * (1 - exp(-2θ*t))
    return exp(-0.5 * (z - μₜ)^2 / sₜ²) / sqrt(2π * sₜ²)
end

catpdf(z,t) = 0.5*happypdf(z, t; θ=1/2, σ=1, λ=0.8) +
              0.2*unhappypdf(z, t; θ=1/2, σ=1, z0=-1.2, s=0.1) +
              0.3*unhappypdf(z, t; θ=1/2, σ=1, z0=-0.8, s=0.1)

lines(-3..3, z->catpdf(z, 0), color=:black, label=L"p_{cat}",
  axis=(title = "Ground truth latent cat distribution",
  xlabel=L"z", xlabelsize=20, ylabel=L"p(z)", ylabelsize=20,
  xticks=([-3,-1.2,-0.8,0.8,3], ["\n-3","😿\n-1.2","😾\n-0.8","😺\n0.8","\n3"]), xticklabelsize=16)
)

#
We have identified the Ornstein-Uhlenbeck process as a candidate for a forward latent "noisification" process.  That is, it takes samples from the latent cat distribution and, as $t \to \infty$ it evolves them to samples from $\mathcal{N}\left(0, \frac{D}{\theta}\right)$.  For the standard normal limit we just need to choose $D$ and $\theta$ so that the variance is one, so let's choose $D = \theta = 1/2$, and so $\sigma = \sqrt{2D} = 1$.

Here's the code to run the simulation forwards in time using a random walk.  We've adapted the code from last lesson to be agnostic to the latent dimension, so we can (next lesson!) use it unchanged for higher-dimensional latent spaces.

In [ ]:
function simulate_sde(; z0, nsteps, T, f = (z,t) -> 0*z, σ = 1)

    Z = stack(fill(z0, nsteps+1))

    dt = T / nsteps  # timestep

    for n = 1:nsteps
        tₙ = (n-1) * dt
        zₙ = selectdim(Z, ndims(Z), n)
        zₙ₊₁ = selectdim(Z, ndims(Z), n+1)
        ξ = randn_like(zₙ)
        dW = sqrt(dt) * ξ
        zₙ₊₁ .= zₙ + f(zₙ,tₙ) * dt + σ * dW
    end

    return Z
end

#
We will draw 10,000 samples from the latent cat distribution.  Again, we emphasise that this step is only possible because we are using this made-up, known, ground truth distribution for cat latents.  We'll overlay the analytic PDE solution to confirm we have it all right.

In [ ]:
nparticles = 10_000
nsteps = 1000
z0 = [rand(Exponential(0.8), nparticles÷2);
      rand(Normal(-1.2, 0.1), nparticles÷5);
      rand(Normal(-0.8, 0.1), 3*nparticles÷10)]
hist!(z0; bins=200, normalization = :pdf)
xlims!(-3,3)
current_figure()

#
The plan then, is for all these "particles" (latent coordinates) to go on a random walk subject to the Ornstein-Uhlenbeck process, in such a way that as $t \to \infty$ they arrange themselves as if they were drawn from $\mathcal{N}(0,1)$.  This is the forward diffusion process.

Now, running the simulation to $t \to \infty$ isn't practical.  So how far is "far enough"?  For today, we will just decide that a maximum simulation time of $T = 5$ is far enough.  In the next lesson we will justify this choice more rigorously.

In [ ]:
T = 5
θ = 1/2
σ = 1
Z = simulate_sde(; z0, nsteps, T, f = (z,t) -> -θ*z, σ = 1)

#
Let's visualise the particle distributions over time.  We'll continue to overlay the analytical solutions.  Remembering we know these curves for any time $t$ because we made this example up in the first place to have an analytic solution.

In [ ]:
fig = Figure(size=(800,800), fontsize=20)
tvec = [0 0.01 0.05 ; 0.1 1 5]
for (pos, t) in pairs(tvec)
    idx = round(Int, nsteps*t/T + 1)
    hist(fig[Tuple(pos)...], Z[:,idx]; bins=100, normalization = :pdf,
         axis = (title = "t = $t", xlabel = L"z", ylabel = L"u(z,t)"))
    lines!(fig[Tuple(pos)...], -3..3, z->catpdf(z, t), color = :black)
    xlims!(-3,3)
end
fig

#
So indeed we observe that the particle distribution under the Ornstein-Uhlenbeck process limits to the standard normal distribution as $t \to \infty$.  All the fine details of the latent cat distribution $p(z)$ are washed out, leaving us with a distribution of particles consistent with $\mathcal{N}(0,1)$.

We have multiple mathematical formulations with which to describe what happened.  In the SDE view, we solved the Ornstein-Uhlenbeck process
$$
\textrm{d}Z_{t} = -\theta Z_t\, \textrm{d}t + \sigma\, \textrm{d}W_t\qquad
$$
with initial positions drawn from the latent distribution
$$
Z_0 \sim p(z)
$$
and found that for $t = T$ sufficiently large, the particles $Z_T$ were distributed according to
$$
Z_T \sim \mathcal{N}(0,1).
$$
The histograms of the particle positions $Z_t$ as they evolve over time provide the blue bars in the figures above.

In the PDE view of things, we solved the equation
$$
\frac{\partial u}{\partial t} = \theta \frac{\partial}{\partial z}(z u) + D \frac{\partial^2 u} {\partial z^2}
$$
subject to the initial condition
$$
u(z,0) = p(z)
$$
and found that for $t = T$ sufficiently large, the distribution approached
$$
u(z,T) = \frac{1}{\sqrt{2\pi}}\exp\left(-\frac{z^2}{2}\right).
$$
The probability density function $u(z,t)$ as it evolves over time is illustrated by the black curves in the figures above.

# "De-Noisifying" a latent

So the forward process of latent image "noisification" works as advertised: the Ornstein-Uhlenbeck process takes an initial latent drawn from the distribution $p(z)$ and evolves it to a draw from $\mathcal{N}(0,1)$.

But what we really want, in order to generate new cat images, is a procedure to reverse this process.  That is, to map a random draw from $\mathcal{N}(0,1)$ to a coordinate which is consistent with the ground truth latent cat distribution.

So what are we waiting for?  Surely that is just running the "noisifying" process in reverse.  We'll set $\tau = T - t$ and substitute into the SDE
$$
\begin{align*}
\textrm{d}Z_{t} &= -\theta Z_t\, \textrm{d}t + \sigma\, \textrm{d}W_t \\
 &= -\theta Z_t\, \textrm{d}t + \sigma\, \sqrt{\textrm{d}t}\, \xi_t
\end{align*}
$$
to find the correct time-reversed form.  Here we go:
$$
\tau = T - t \iff t = T - \tau \implies \textrm{d}t = - \textrm{d}\tau
$$
so our time-reversed equation is
$$
\begin{align*}
\textrm{d}Z_{\tau} &= \theta Z_{\tau}\, \textrm{d}\tau + \sigma\, \sqrt{-\textrm{d}\tau}\, \xi_\tau
\end{align*}
$$
subject to
$$
Z_{\tau = 0} \sim \mathcal{N}(0,1)
$$
which we would solve from $\tau = 0$ to $\tau = T$. Job done, right?

Wait, is that a $\sqrt{-\textrm{d}\tau}$ factor?  That can't be good!  We seem to have hit a roadblock with this idea.  There's no such thing as $\sqrt{-\textrm{d}\tau}$, so this idea looks like a non-starter.

So that's it.  There's no reverse diffusion, and our dream of generating new cat images is over.



# "De-Noisifying" a latent -- try again maybe?

Wait, so far we've only looked at the SDE interpretation of the "noisification" process. There is also the PDE interpretation, via the Fokker-Planck equation.  Remember the correspondence:
$$
\textrm{d}Z_{t} = f(Z_t, t)\, \textrm{d}t + \sigma\, \textrm{d}W_t \quad \Longleftrightarrow \quad \frac{\partial u}{\partial t} = -\frac{\partial}{\partial z}(f u) + D \frac{\partial^2 u} {\partial z^2}\qquad (*)
$$
where $D = \sigma^2 / 2$.

For our Ornstein-Uhlenbeck process this means
$$
\textrm{d}Z_{t} = -\theta Z_t\, \textrm{d}t + \sigma\, \textrm{d}W_t \quad \Longleftrightarrow \quad  \frac{\partial u}{\partial t} = \frac{\partial}{\partial z}(\theta z u) + D \frac{\partial^2 u} {\partial z^2}\,.
$$

So let's run time backwards in the PDE instead.  Set $\tau = T - t$ again, and all that happens is we get a sign flip:
$$
\frac{\partial u}{\partial \tau} = -\frac{\partial}{\partial z}(\theta z u) - D \frac{\partial^2 u} {\partial z^2}\,.
$$
There's no pesky $\sqrt{-\textrm{d}\tau}$ to worry about!  Now we can convert back to the corresponding SDE and get our random walk simulations up and running.

Using the correspondence $(*)$ we can pattern-match to deduce we need to choose $f(z,t) = \theta z$, and $\sigma = \sqrt{-2D}$.

Oh.

You _really_ can't run diffusion in reverse.  Every time you try, the maths tells you no.

# "De-Noisifying" a latent -- one more try?

OK, let's give this one more try before we abandon all hope.  We _can_ reverse time in the Fokker-Planck equation.  The problem comes when we try to map the resulting equation back to an SDE: the diffusion term doesn't work because there's no way to choose $\sigma$ to satisfy $\sigma^2/2 = -D$.

So...what if there was no diffusion term?  Sounds crazy right?  There _is_ a diffusion term.  We can't just magic it away.

And for sure, there is a negative diffusion term in our time-reversed equation as it stands.
$$
\frac{\partial u}{\partial \tau} = -\frac{\partial}{\partial z}(\theta z u) - D \frac{\partial^2 u} {\partial z^2}\,.
$$

But perhaps we can rewrite the equation to make it go away.

$$
\begin{align*}
\frac{\partial u}{\partial \tau} &= -\frac{\partial}{\partial z}(\theta z u) - D \frac{\partial} {\partial z} \frac{\partial u} {\partial z}\\
&= -\frac{\partial}{\partial z}(\theta z u) - D \frac{\partial} {\partial z} \left(\frac{\partial \log u} {\partial z}\, u\right)\quad (\textrm{nice trick to introduce factor of u)}\\
&= \frac{\partial}{\partial z}\left(-\theta z u - D \frac{\partial \log u} {\partial z}\, u\right) \\
&= \frac{\partial}{\partial z}\left(\left[-\theta z - D \frac{\partial \log u} {\partial z}\right]\, u\right) \\
&= \frac{\partial}{\partial z}\left(\left[-\theta z - \frac{\sigma^2}{2} \frac{\partial \log u} {\partial z}\right]\, u\right) \\
&= -\frac{\partial}{\partial z}\left(\left[\theta z + \frac{\sigma^2}{2} \frac{\partial \log u} {\partial z}\right]\, u\right)
\end{align*}
$$
Tada!  No diffusion term left, only a wacky-looking drift term.  Pattern-matching using $(*)$ again, we can convert this back to an SDE, arriving at
$$
\textrm{d} Z_\tau = \left(\theta Z_\tau + \frac{\sigma^2}{2} \frac{\partial \log u} {\partial z}\right) \textrm{d}\tau\,.
$$

And thus, we have pulled off one of the greatest swindles -- a time-reversed diffusion equation in SDE form.  In fact, it's not _even_ an SDE -- there's no diffusion term here!  This is a pure _ordinary_ differential equation:
$$
\frac{\textrm{d}z}{\textrm{d}\tau} = \theta z + \frac{\sigma^2}{2} \frac{\partial \log u} {\partial z}\,,
$$
a fully _deterministic_ model to reverse the Ornstein-Uhlenbeck process.

Does it work?

Well yes.  But it's a total cheat!

The drift term requires that we plug in the gradient of the log probability density, $\partial \log u/\partial z$.  In other words, we can reverse the diffusion process...provided we have complete knowledge of the probability density function for all time!

Still, it is quite satisfying to watch this reverse process work its magic.  Let's give it a try.

# Demonstrating reverse diffusion
We need to define the log probability density $\partial \log u/\partial z$. This is sometimes called the [score function](https://en.wikipedia.org/wiki/Score_(statistics)) in statistics, so we'll call it that too.  I certainly don't fancy differentiating the complicated PDF by hand, but that's what automatic differentiation is for!  We have only one variable, so forward mode AD is the ideal tool for the job.

In [ ]:
score = (z,t) -> derivative(z->catpdf(z,t), AutoForwardDiff(), z) / catpdf(z,t) # d/dz log(u) = du/dz / z

#
We start the reverse process from a big bunch of standard normal samples
$$
Z_{\tau = 0} \sim \mathcal{N}(0,1).
$$

In [ ]:
z = randn(nparticles)  # standard normal initial distribution

#
Keeping in mind our variable substitution $t = T - \tau$, the equation to evolve the samples in time is
$$
\frac{\textrm{d}z}{\textrm{d}\tau} = \theta z + \frac{\sigma^2}{2} \frac{\partial \log u(z,T-\tau)} {\partial z}\,,\quad  0 < \tau \leq T.
$$

Even though we're now solving an ODE, we can carry on using our SDE simulation code, just set $\sigma = 0$.  The numerical scheme thus collapses to Euler's method, which you'll recall fondly from MXB103.  (You could absolutely do better by coding up a second order method.)

The output below shows the full history of the reverse simulation.  The final column corresponds to $\tau = T$, i.e. $t = 0$.  At this final point of the reverse diffusion process, the samples should all have evolved, as if by magic, to follow the latent cat distribution.

In [ ]:
Zrev = simulate_sde(; z0 = z, nsteps = 1000, T=5, f = (z,τ) -> z/2 + score.(z,5-τ)/2, σ = 0) # n.b. score(z, T-τ)

#
Here's the figure that confirms we have it working correctly.  The histogram of the final particle positions matches the ground truth distribution.

In [ ]:
hist(Zrev[:,end], bins=200, normalization=:pdf,
     axis=(title = "Ground truth latent cat distribution",
     xlabel=L"z", xlabelsize=20, ylabel=L"u(z)", ylabelsize=20,
     xticks=([-3,-1.2,-0.8,0.8,3], ["\n-3","😿\n-1.2","😾\n-0.8","😺\n0.8","\n3"]), xticklabelsize=16))
lines!(-3..3, z->catpdf(z, 0), color=:black)
xlims!(-3, 3)
current_figure()

#
So in summary, we solved the ODE
$$
\frac{\textrm{d}z}{\textrm{d}\tau} = \theta z + \frac{\sigma^2}{2} \frac{\partial \log u(z,T-\tau)} {\partial z}\,,\quad  0 < \tau \leq T
$$
subject to
$$
z_{\tau = 0} \sim \mathcal{N}(0,1)
$$
and confirmed visually that
$$
z_{\tau = T} \sim p(z)\,.
$$

Hence, after much hard work, we appear to have devised a method of _sampling_ from the ground truth latent distribution $p(z)$ by evolving an initial sample from the standard normal distribution.

# Cat sampler
Let's even define a convenience "cat sampler" function.  Just as `randn` samples from a normal distribution, so `randcats` will sample from the latent cat distribution.

With this function, we can easily sample as many latent cat coordinates as we like.

In [ ]:
function randcats(ncats; nsteps = 1000, T = 5)
    Z = simulate_sde(; z0 = randn(ncats), nsteps, T, f = (z,τ) -> z/2 + score.(z,T-τ)/2, σ = 0)
    return Z[:,end]  # just return the solution at the final time
end

cats = sort(randcats(10))  # here are 10 sorted random cat latents

#
You can see the samples (red dots) correctly avoid the region of "angry, happy" latent space $(-0.5, 0)$, which we know contains no cats.

In [ ]:
scatter!(cats, zero(cats), color=:red)
current_figure()

# But is this just cheating?
As we've already pointed out though, this all seems like a bit of a cheat.  It only works because we've coded in our knowledge of the ground truth distribution, via the score function $\partial \log u /\partial z$.

But perhaps, if we were given a great many cat pictures to _learn_ from initially, we could _train_ a model that could act as a surrogate for the real score function.  That is, we would try to learn some kind of approximate score function from data first, then we could use that trained model in place of the true score function, and generate new cat pictures that way.  It might not be perfect, but just maybe it would be good enough.

Oh, and it would be in higher dimensional space! Real latent variables aren't just scalars -- they're vectors (or higher).  So we'll also need to generalise what we've done today to higher dimensions.  On to the next lesson!

# Summary

In this lesson we learned:

* the Ornstein-Uhlenbeck process for forward diffusion in which the particle distribution limits to Gaussian

* the difficulties of reversing a diffusion process owing to the stochastic variability

* how to work around this difficulity with a deterministic process involving the score function

* that to make further progress, we will need a way to learn about this score function from data